![Universidad Espíritu Santo](https://raw.githubusercontent.com/andresrubiop/miar0525-estudiantes/main/utils/logo-uees-color.png)

<div style="background:#821436;color:#FFFFFF;padding:14px 18px;border-radius:10px;margin:6px 0 12px 0"><div style="font-size:12px;letter-spacing:.08em;text-transform:uppercase;opacity:.9">Aprendizaje Automático · MIAR0525 · Semana 2 · Tarea 2 · 20 % · entrega hasta el vie 02/10, 23:59</div><div style="font-size:22px;font-weight:700;margin-top:4px">T2 · Comparación reproducible de modelos supervisados</div><div style="font-size:12px;opacity:.9;margin-top:4px">Postgrado · Maestría en Inteligencia Artificial · UEES</div></div>

| | |
|---|---|
| **Qué entregas** | **Este mismo notebook**, ejecutado de principio a fin y guardado con sus salidas, renombrado `T2_Apellido_Nombre.ipynb`. No hay informe en PDF ni hay que subir `mlflow.db`: la evidencia de MLflow es la tabla de corridas impresa en la sección 3. |
| **Caso** | Un equipo de calidad de software quiere priorizar la revisión de código: predecir qué módulos tienen defectos. |
| **Datos** | jm1 (OpenML `data_id=1053`): 10 885 módulos, 21 métricas de código, ≈ 19 % defectuosos. **Trampa:** tiene duplicados y valores inconsistentes documentados. |
| **Resultado de aprendizaje** | RDA1 · competencias CG-G1 y CE-G1 |
| **Puedes reutilizar** | E2.1 (regularización) · E2.2 (logística y calibración) · E2.3 (SVM y KNN) · E2.4 (árboles, bosques, comparación y MLflow) |

## Cómo se califica

Cada sección es un criterio de la rúbrica y lleva su puntaje en el título. En cada una hay **Qué hacer**, celdas de
código con `# TODO` y una celda **Tu análisis** que debes responder: *el código que corre sin análisis no suma puntos.*

**Uso de IA.** Permitida (agéntica o de chat) **si la declaras en la sección 7**, indicando en qué secciones la usaste.
La rúbrica evalúa tus decisiones, su justificación y cómo verificaste los resultados.

**Antes de entregar**: *Kernel → Restart & Run All*, revisa que todas las celdas tengan salida y guarda con tu nombre.

## 0 · Configuración y datos (obligatorio)

In [ ]:
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from cycler import cycler
from sklearn.datasets import fetch_openml

warnings.filterwarnings("ignore", category=FutureWarning)
SEED = 2026
UEES = {"vino": "#821436", "azul": "#1F6F8B", "ocre": "#C28E0E", "verde": "#3A7D44", "gris": "#77787B"}
plt.rcParams.update({
    "axes.prop_cycle": cycler(color=list(UEES.values())), "axes.titlecolor": UEES["vino"],
    "axes.titleweight": "bold", "axes.edgecolor": UEES["gris"], "axes.grid": True, "grid.color": "#EEE8EA",
    "axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 110, "legend.frameon": False,
})
print(f"Python {sys.version.split()[0]} · scikit-learn {sklearn.__version__} · semilla {SEED}")

jm1 = fetch_openml(data_id=1053, as_frame=True)
X = jm1.data.apply(pd.to_numeric, errors="coerce")
y = (jm1.target.astype(str).str.lower().isin(["true", "1", "yes"])).astype(int)
print(f"{len(X)} módulos · {X.shape[1]} métricas · {y.mean():.1%} con defectos · faltantes: {int(X.isna().sum().sum())}")
X.head(3)

## 1 · Limpieza justificada y partición sin fuga · 20 puntos

**Qué hacer:**

1. Documenta los problemas del conjunto: **duplicados** (filas idénticas), faltantes y valores imposibles
   (por ejemplo, métricas negativas o líneas de código en cero).
2. Decide qué haces con cada uno **y justifícalo**. Si eliminas duplicados, hazlo **antes** de partir, para que el
   mismo módulo no quede en entrenamiento y prueba.
3. Separa la prueba (20 %, estratificada) y no la toques hasta la sección 5.

*Reutiliza:* el reporte de calidad de E1.3 y la limpieza de E2.4.

In [ ]:
from sklearn.model_selection import train_test_split

# TODO: reporte de duplicados, faltantes y valores imposibles; limpieza justificada.


# TODO: partición estratificada (20 %, random_state=SEED) después de la limpieza.
# X_train, X_test, y_train, y_test = ...

In [ ]:
# Autoverificación de la sección 1
assert "X_train" in dir() and "X_test" in dir(), "Falta la partición."
assert abs(y_train.mean() - y_test.mean()) < 0.02, "La partición no parece estratificada."
print("✓", len(X_train), "entrenamiento ·", len(X_test), "prueba ·", f"{y_train.mean():.1%} defectuosos")

**Tu análisis (sección 1).** ¿Cuántos duplicados había y qué hiciste con ellos? ¿Por qué dejar duplicados entre
entrenamiento y prueba inflaría los resultados?

*(escribe aquí)*

## 2 · Cinco modelos con su pipeline · 20 puntos

**Qué hacer:** arma un `Pipeline` para cada uno de los cinco modelos, con el preprocesamiento adentro
(imputación y escalado donde haga falta):

1. Regresión logística
2. SVM (`SVC`)
3. Árbol de decisión
4. Random forest
5. `HistGradientBoostingClassifier` (como referencia)

Agrega también un `DummyClassifier` como modelo de referencia trivial.

*Reutiliza:* los pipelines de E2.2, E2.3 y E2.4.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

# TODO: define el diccionario modelos = {"baseline": ..., "logística": ..., "svm": ..., "árbol": ...,
#       "bosque": ..., "boosting": ...}, cada uno como Pipeline.
# modelos = {...}

In [ ]:
# Autoverificación de la sección 2
assert len(modelos) >= 6, "Faltan modelos (cinco más el de referencia)."
assert all(hasattr(m, "fit") for m in modelos.values()), "Todos deben ser estimadores de scikit-learn."
print("✓ modelos definidos:", ", ".join(modelos))

**Tu análisis (sección 2).** ¿Qué modelos necesitan escalado y cuáles no? ¿Por qué el preprocesamiento va dentro
del pipeline y no antes?

*(escribe aquí)*

## 3 · Búsqueda de hiperparámetros sin fuga y registro en MLflow · 15 puntos

**Qué hacer:**

1. Define una malla pequeña y razonable para al menos tres de los modelos (justifica los rangos).
2. Busca con `GridSearchCV` o `RandomizedSearchCV` **dentro del entrenamiento**, con validación cruzada
   estratificada de 5 folds y la misma semilla para todos.
3. Registra **cada corrida** en MLflow (`sqlite:///mlflow.db`): parámetros, métrica media y desviación.
4. Imprime al final la tabla de corridas con `mlflow.search_runs()`: **esa tabla es tu evidencia**, no hace falta
   subir la base de datos.

*Reutiliza:* el bloque de MLflow de E2.4 · Nivel 3.

In [ ]:
import mlflow
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_validate

mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("T2-jm1")
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

# TODO: búsqueda de hiperparámetros por modelo y registro de cada corrida en MLflow.


# TODO: imprime la tabla de corridas (mlflow.search_runs()) con parámetros y métricas.

**Tu análisis (sección 3).** ¿Por qué el mejor puntaje de la búsqueda es optimista? ¿Qué rangos elegiste y por qué?

*(escribe aquí)*

## 4 · Desbalance y umbral · 10 puntos

**Qué hacer:**

1. Trata el desbalance de dos formas y compáralas: `class_weight="balanced"` y el **umbral** elegido por validación.
2. Define el costo de cada error en este caso (no revisar un módulo defectuoso frente a revisar uno sano) y elige
   el umbral que minimiza el costo esperado, **sin usar la prueba**.

*Reutiliza:* el umbral por costo de E1.2 y el tratamiento del desbalance de E2.2.

In [ ]:
from sklearn.model_selection import cross_val_predict

# TODO: costos del caso, probabilidades fuera de muestra, barrido de umbrales y elección justificada.
# COSTO_FP, COSTO_FN = ..., ...
# UMBRAL = ...

**Tu análisis (sección 4).** ¿Repesar las clases y mover el umbral dan el mismo resultado? ¿Cuál prefieres aquí
y por qué?

*(escribe aquí)*

## 5 · Curvas de aprendizaje, calibración y evaluación final · 20 puntos

**Qué hacer:**

1. Grafica **curvas de aprendizaje** de al menos dos modelos e interprétalas (¿sesgo o varianza?).
2. Calibra el modelo elegido y muestra su **diagrama de fiabilidad** antes y después.
3. Evalúa **una sola vez** en la prueba con tu umbral: matriz de confusión, precisión, recall, F1, AUC y costo.

*Reutiliza:* las curvas de aprendizaje de E2.4 y la calibración de E2.2 · Nivel 3.

In [ ]:
from sklearn.calibration import CalibratedClassifierCV, CalibrationDisplay
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.model_selection import learning_curve

# TODO: curvas de aprendizaje, calibración y evaluación final en la prueba.

**Tu análisis (sección 5).** ¿Qué dicen las curvas: más datos o más capacidad? ¿El modelo estaba bien calibrado?
¿Los resultados en la prueba se parecen a los de la validación cruzada?

*(escribe aquí)*

## 6 · Tabla comparativa, recomendación y limitaciones · 15 puntos

**Qué hacer:**

1. Arma una tabla final con los seis modelos: **media ± desviación** de las métricas y el costo en el punto de
   operación elegido.
2. Recomienda uno, en 150–250 palabras, como si escribieras al líder del equipo de calidad: cuánto mejora sobre el
   modelo de referencia, qué cuesta cada error, qué tan estable es la diferencia frente a los otros modelos y qué
   limitaciones tiene.

In [ ]:
# TODO: tabla comparativa final de los seis modelos.

**Tu recomendación (sección 6).**

*(escribe aquí)*

## 7 · Declaración de uso de IA (obligatoria)

Completa la tabla. Si no usaste IA, escribe "No usé IA" y firma igual (norma f del sílabo).

| | |
|---|---|
| **Herramientas** | *(por ejemplo: ChatGPT, Claude Code; o "ninguna")* |
| **Secciones donde la usé** | *(por ejemplo: sección 3 para la malla y sección 5 para los gráficos)* |
| **Para qué** | *(escribir código, depurar, redactar el análisis, revisar mi interpretación…)* |
| **Prompts relevantes** | *(pega los 2 o 3 más importantes)* |
| **Qué verifiqué yo** | *(ejecuté todo de cero, comparé las cifras con MLflow, revisé que no haya fuga…)* |
| **Qué corregí o descarté** | *(qué propuso la IA que no usaste y por qué)* |

**Autoría.** El análisis, las decisiones y las conclusiones de este notebook son míos.

Nombre: *(tu nombre)* · Fecha: *(fecha de entrega)*

## Lista de cotejo antes de entregar

- [ ] Reinicié el kernel y ejecuté todo de arriba hacia abajo, sin errores.
- [ ] Los duplicados se trataron **antes** de partir los datos.
- [ ] La búsqueda de hiperparámetros nunca tocó la prueba.
- [ ] La tabla de corridas de MLflow aparece impresa en la sección 3.
- [ ] Reporté media ± desviación y no declaré ganador por diferencias menores que la variabilidad.
- [ ] Respondí todas las celdas "Tu análisis" y completé la declaración de uso de IA.
- [ ] Guardé el archivo como `T2_Apellido_Nombre.ipynb` y lo subí al LMS.

In [ ]:
from datetime import datetime

print(f"Notebook ejecutado el {datetime.now():%Y-%m-%d %H:%M} · scikit-learn {sklearn.__version__} · semilla {SEED}")